# AlphaGenome Fine-Tuning Notebook

This notebook is designed for practical, editable fine-tuning runs.

Workflow:
1. Set paths and training options
2. (Optional) Download files
3. Validate inputs
4. Build heads and intervals
5. Initialize model
6. Train and checkpoint


## 0) Install and import dependencies

In [ ]:
# Installation will take a few minutes
%pip install -q "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
%pip install -q git+https://github.com/google-deepmind/alphagenome.git
%pip install -q git+https://github.com/google-deepmind/alphagenome_research.git
%pip install -q alphagenome-ft
print("Done.")

In [1]:
from __future__ import annotations

from pathlib import Path
import gzip
import shutil
import urllib.request

from alphagenome_ft import create_model_with_heads
from alphagenome_ft.finetune.config import load_targets_config, prepare_head_specs
from alphagenome_ft.finetune.data import prepare_intervals_from_fold, BigWigDataModule, build_fasta_index
from alphagenome_ft.finetune.train import register_predefined_heads, train

/grid/koo/home/nagai/projects/alphagenome_collab/.venv/lib/python3.12/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.9.0 is installed, but it is not compatible with the installed jaxlib version 0.9.0.1, so it will not be used.
  warnings.warn(
/grid/koo/home/nagai/projects/alphagenome_collab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) User Config

Edit these values first. Keep paths relative to your project root if possible.

In [2]:
# Core input files
PROJECT_DIR = Path("./")  # Data directory
FASTA_PATH = PROJECT_DIR / "hg38.fa"  # Change the path or optional download (see below)
TARGETS_CONFIG_PATH = PROJECT_DIR / "targets_rna_demo.yaml"  # Change the path or optional download (see below)

# Data options
FOLD = "1"  # 0, 1, 2, 3
WINDOW_SIZE = 1_048_576  # 1 Mbp window

# Model options
ORGANISM = "HOMO_SAPIENS"
MODEL_VERSION = "fold_1"  # fold_0, fold_1, fold_2, fold_3, all_folds
HEADS_ONLY = True

# Training options
BATCH_SIZE = 1
NUM_EPOCHS = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
SEED = 42
SHUFFLE = True
DROP_LAST = False
MAX_TRAIN_STEPS = None
VERBOSE = True

# Checkpointing / early stopping
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints/finetune_demo_notebook"
BEST_METRIC = "valid_loss"
BEST_METRIC_MODE = "min"
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.0


## 2) Optional Data Download

Download FASTA and the 3 RNA demo files from `alphagenome_ft/data` on GitHub when missing.


In [3]:
ENABLE_DATA_DOWNLOAD = True
ENABLE_FASTA_DOWNLOAD = True

GITHUB_DATA_BASE = "https://raw.githubusercontent.com/genomicsxai/alphagenome_ft/main/data"
DEMO_FILES = [
    "targets_rna_demo.yaml",
    "example_rna_rep1_chr1.bw",
    "example_rna_rep2_chr1.bw",
]
FASTA_DOWNLOAD_URL = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

if ENABLE_DATA_DOWNLOAD:
    for filename in DEMO_FILES:
        dst = PROJECT_DIR / filename
        if dst.exists():
            print(f"Skip existing: {dst}")
            continue
        src = f"{GITHUB_DATA_BASE}/{filename}"
        print(f"Downloading: {src} -> {dst}")
        urllib.request.urlretrieve(src, dst)
else:
    print("Core data download is disabled.")

if ENABLE_FASTA_DOWNLOAD:
    if FASTA_PATH.exists():
        print(f"Skip existing: {FASTA_PATH}")
    else:
        gz_path = FASTA_PATH.with_suffix(FASTA_PATH.suffix + ".gz")
        print(f"Downloading: {FASTA_DOWNLOAD_URL} -> {gz_path}")
        urllib.request.urlretrieve(FASTA_DOWNLOAD_URL, gz_path)
        print(f"Decompressing: {gz_path} -> {FASTA_PATH}")
        with gzip.open(gz_path, "rb") as src, FASTA_PATH.open("wb") as dst:
            shutil.copyfileobj(src, dst)
        gz_path.unlink()
else:
    print("FASTA download is disabled.")

FASTA_INDEX_PATH = Path(f"{FASTA_PATH}.fai")
if FASTA_PATH.exists() and not FASTA_INDEX_PATH.exists():
    print(f"Building FASTA index: {FASTA_INDEX_PATH}")
    build_fasta_index(FASTA_PATH)
elif FASTA_INDEX_PATH.exists():
    print(f"Found FASTA index: {FASTA_INDEX_PATH}")
else:
    print("Skip FASTA indexing because FASTA is missing.")

Downloading: https://raw.githubusercontent.com/genomicsxai/alphagenome_ft/main/data/targets_rna_demo.yaml -> targets_rna_demo.yaml
Downloading: https://raw.githubusercontent.com/genomicsxai/alphagenome_ft/main/data/example_rna_rep1_chr1.bw -> example_rna_rep1_chr1.bw
Downloading: https://raw.githubusercontent.com/genomicsxai/alphagenome_ft/main/data/example_rna_rep2_chr1.bw -> example_rna_rep2_chr1.bw
Downloading: https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz -> hg38.fa.gz
Decompressing: hg38.fa.gz -> hg38.fa
Building FASTA index: hg38.fa.fai


## 3) Validate Inputs

In [4]:
for path in [FASTA_PATH, FASTA_INDEX_PATH, TARGETS_CONFIG_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

if CHECKPOINT_DIR is not None:
    CHECKPOINT_DIR = Path(CHECKPOINT_DIR).resolve()
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Input validation passed.")

Input validation passed.


## 4) Build Head Specs and Fold Intervals


In [5]:
# If you pass ``base_dir``, paths in the config can be relative to it.
# Otherwise, they will be resolved relative to the current working directory.
config_dict = load_targets_config(
    TARGETS_CONFIG_PATH,
    base_dir=PROJECT_DIR
)
# print(config_dict)

# Build head specifications from the config.
head_specs = prepare_head_specs(config_dict)

register_predefined_heads(head_specs)  # if using custome heads, they must be registered independently

# Create dataframe of intervals for the specified fold and window size
intervals = prepare_intervals_from_fold(
    fold=FOLD,
    window_size=WINDOW_SIZE,
    organism=ORGANISM,
)

print("Heads:", [spec.head_id for spec in head_specs])
for split in ("train", "valid", "test"):
    print(f"{split}: {len(intervals.get(split, []))} intervals")

Filtered out 1265 intervals from valid/test sets due to training overlap.
Heads: ['my_rna_head']
train: 42258 intervals
valid: 5877 intervals
test: 6097 intervals


## 5) Initialize or Resume Model

### Kaggle credentials (required for downloading AlphaGenome weights)

The next cell downloads the base Alphagenome model weights from Kaggle when creating the encoder fine-tuning model. 

**One-time setup:**

1. **Accept the dataset agreement:** [AlphaGenome on Kaggle](https://www.kaggle.com/models/google/alphagenome) → open the page and accept terms if prompted.
2. **Create a _legacy_ API token:** [Kaggle Account → Create New Token](https://www.kaggle.com/settings) (downloads `kaggle.json`). Use the **username** and **key** from that file (the **key** is the long token, not your account password). 

**NOTE** - do not use a 'API Tokens (Recommended)' as this will **not** work.

Once you log in and it's successful, rerun the code to create the model.

In [6]:
model = create_model_with_heads(
    MODEL_VERSION,
    heads=[spec.head_id for spec in head_specs],
    detach_backbone=HEADS_ONLY
)

print("Model ready.")

Loading pretrained AlphaGenome model...

/grid/koo/home/nagai/projects/alphagenome_collab/.venv/lib/python3.12/site-packages/pyfaidx/__init__.py:589: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/grid/koo/home/nagai/projects/alphagenome_collab/.venv/lib/python3.12/site-packages/pyfaidx/__init__.py:589: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


✓ Pretrained model loaded
Initializing heads: ['my_rna_head']
Initializing parameters... (seq_len=16384)
✓ Head parameters initialized
Merging pretrained backbone with heads...
✓ Parameters merged
✓ Model created successfully
  Total parameters: 450,471,061
  Heads: ['my_rna_head']
Model ready.


## 6) Build Data Module

In [7]:
data_module = BigWigDataModule(
    intervals=intervals,
    fasta_path=FASTA_PATH,
    head_specs=head_specs,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE,
    drop_last=DROP_LAST,
)

print("Data module ready.")

Filtered out 49863 intervals to match BigWig chromosome coverage (removed chromosomes: chr10, chr11, chr12, chr13, chr14, chr15, chr16, chr17, chr18, chr19, chr2, chr20, chr21, chr22, chr3, chr4, chr5, chr6, chr7, chr8, chr9, chrX; removed intervals: train=38327, valid=5676, test=5860).
Data module ready.


## 7) (Optional) Smoke Test Settings

Use this to test your setup quickly before a long run.

In [8]:
SMOKE_TEST = True
if SMOKE_TEST:
    NUM_EPOCHS = 1
    MAX_TRAIN_STEPS = 20
    print("Smoke test enabled.")
else:
    print("Smoke test disabled.")


Smoke test enabled.


## 8) Train

In [9]:
train(
    model,
    data_module,
    head_specs,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_epochs=NUM_EPOCHS,
    seed=SEED,
    max_train_steps=MAX_TRAIN_STEPS,
    heads_only=HEADS_ONLY,
    checkpoint_dir=CHECKPOINT_DIR,
    organism=ORGANISM,
    best_metric=BEST_METRIC,
    best_metric_mode=BEST_METRIC_MODE,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    verbose=VERBOSE,
)

JIT-compiling step functions (first call will be slow)...
Train plan: 3931 examples | 3931 step(s)/epoch | 1 epoch(s) | total step(s) 20

Epoch 1/1


  step 0020/3931 | loss 5.55867
  Train loss: 3.9591
  Validation metrics: my_rna_head=7.6937
  Metric improved (valid_loss = 7.6937)  -- saving best checkpoint
✓ Checkpoint saved to /grid/koo/home/nagai/projects/alphagenome_collab/alphagenome_ft/notebooks/checkpoints/finetune_demo_notebook/best
  Saved: Heads only ['my_rna_head']
✓ Checkpoint saved to /grid/koo/home/nagai/projects/alphagenome_collab/alphagenome_ft/notebooks/checkpoints/finetune_demo_notebook/last
  Saved: Heads only ['my_rna_head']
  Reached requested training steps: 20/20

Training complete!
